# Insurance Documents Retrieval Testing

This notebook helps you test and explore your Qdrant vector database.

**What you can do:**
- Connect to Qdrant and inspect collections
- Test semantic search queries
- Compare results across different embedding models
- Visualize chunks and metadata
- Debug retrieval quality

## 1. Setup & Connection

In [ ]:
import sys
sys.path.append('..')

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from src.config import QDRANT_HOST, OPENAI_API_KEY
import pandas as pd
import json
from pprint import pprint

print(f"✅ Connecting to Qdrant at: {QDRANT_HOST}")

In [ ]:
# Initialize Qdrant client
client = QdrantClient(url=QDRANT_HOST)

# List all collections
collections = client.get_collections()
print("\n📊 Available Collections:")
print("=" * 70)
for col in collections.collections:
    info = client.get_collection(col.name)
    print(f"  • {col.name}")
    print(f"    Points: {info.points_count:,}")
    print(f"    Vectors: {info.vectors_count:,}")
    print()

## 2. Select Collection to Test

In [ ]:
# Choose which collection to test (change this to your collection name)
COLLECTION_NAME = "insurance_docs_text-embedding-3-large"

# Get collection info
try:
    info = client.get_collection(COLLECTION_NAME)
    print(f"✅ Using collection: {COLLECTION_NAME}")
    print(f"   Total chunks: {info.points_count:,}")
except Exception as e:
    print(f"❌ Collection not found: {COLLECTION_NAME}")
    print(f"   Available collections: {[c.name for c in collections.collections]}")

## 3. Inspect Sample Data

In [ ]:
# Get sample points to see metadata structure
sample = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=5,
    with_payload=True,
    with_vectors=False  # Don't load vectors (too large)
)

print("📄 Sample Chunks:")
print("=" * 70)
for i, point in enumerate(sample[0], 1):
    print(f"\nChunk {i}:")
    print(f"  Insurance: {point.payload.get('metadata', {}).get('insurance_provider', 'N/A')}")
    print(f"  Company: {point.payload.get('metadata', {}).get('company_display_name', 'N/A')}")
    print(f"  Doc Type: {point.payload.get('metadata', {}).get('document_type', 'N/A')}")
    print(f"  Doc Name: {point.payload.get('metadata', {}).get('document_name', 'N/A')}")
    print(f"  Headers: {point.payload.get('metadata', {}).get('header_1', '')} > {point.payload.get('metadata', {}).get('header_2', '')}")
    print(f"  Preview: {point.payload.get('page_content', '')[:150]}...")

In [ ]:
# Analyze metadata distribution
print("📊 Analyzing Database Statistics...\n")

# Get all points (or sample for large databases)
all_points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10000,  # Adjust if you have more chunks
    with_payload=True,
    with_vectors=False
)

# Extract metadata into DataFrame
data = []
for point in all_points:
    meta = point.payload.get('metadata', {})
    data.append({
        'insurance': meta.get('insurance_provider', 'N/A'),
        'company': meta.get('company_display_name', 'N/A'),
        'doc_type': meta.get('document_type', 'N/A'),
        'is_webpage': meta.get('is_webpage', False),
        'chunk_length': len(point.payload.get('page_content', ''))
    })

df = pd.DataFrame(data)

print("\n📈 Chunks per Insurance Provider:")
print(df['insurance'].value_counts())

print("\n📈 Chunks per Document Type:")
print(df['doc_type'].value_counts())

print("\n📈 Chunk Size Statistics:")
print(df['chunk_length'].describe())

## 4. Setup Embedder for Queries

In [ ]:
# Initialize embedder (must match the model used for indexing)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=OPENAI_API_KEY
)

# Create LangChain vector store for easy querying
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    retrieval_mode=RetrievalMode.HYBRID,  # Use hybrid if indexed with sparse
)

print("✅ Embedder ready for queries")

## 5. Test Queries

In [ ]:
def search(query, k=5, filters=None):
    """Search with optional metadata filters"""
    
    print(f"🔍 Query: {query}")
    if filters:
        print(f"   Filters: {filters}")
    print("=" * 70)
    
    # Search using LangChain (easier)
    if filters:
        results = vector_store.similarity_search_with_score(
            query, 
            k=k,
            filter=filters
        )
    else:
        results = vector_store.similarity_search_with_score(query, k=k)
    
    # Display results
    for i, (doc, score) in enumerate(results, 1):
        meta = doc.metadata
        print(f"\n{i}. Score: {score:.4f}")
        print(f"   Insurance: {meta.get('insurance_provider', 'N/A')}")
        print(f"   Company: {meta.get('company_display_name', 'N/A')}")
        print(f"   Doc Type: {meta.get('document_type', 'N/A')}")
        print(f"   Doc Name: {meta.get('document_name', 'N/A')}")
        print(f"   Headers: {meta.get('header_1', '')} > {meta.get('header_2', '')} > {meta.get('header_3', '')}")
        print(f"   Preview: {doc.page_content[:200]}...")
    
    return results

In [ ]:
# Test Query 1: General search
results = search("What is covered for pregnancy and childbirth?", k=5)

In [ ]:
# Test Query 2: Search with filter (specific insurance)
results = search(
    "dental care coverage",
    k=5,
    filters={"insurance_provider": "goudse_expat_pakket"}
)

In [ ]:
# Test Query 3: Search in specific document type
results = search(
    "premium cost",
    k=5,
    filters={"document_type": "premiums"}
)

In [ ]:
# Test Query 4: Search only webpages (latest info)
results = search(
    "contact information customer service",
    k=5,
    filters={"is_webpage": True}
)

## 6. Compare Different Insurances

In [ ]:
# Compare the same query across different insurance providers
query = "What are the exclusions?"
insurances = ["goudse_expat_pakket", "allianz_care", "cigna_global_care"]

print(f"🔍 Comparing query: '{query}'\n")
print("=" * 70)

for insurance in insurances:
    print(f"\n📋 {insurance.replace('_', ' ').title()}:")
    print("-" * 70)
    
    try:
        results = vector_store.similarity_search_with_score(
            query,
            k=3,
            filter={"insurance_provider": insurance}
        )
        
        for i, (doc, score) in enumerate(results, 1):
            print(f"  {i}. Score: {score:.4f}")
            print(f"     {doc.page_content[:150]}...\n")
    except Exception as e:
        print(f"  ❌ No results for {insurance}")

## 7. Visualize Chunk Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Plot 1: Chunks per insurance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Insurance distribution
df['insurance'].value_counts().plot(kind='barh', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Chunks per Insurance Provider', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Number of Chunks')

# Document type distribution
df['doc_type'].value_counts().plot(kind='bar', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Chunks per Document Type', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Document Type')
axes[0, 1].set_ylabel('Number of Chunks')
axes[0, 1].tick_params(axis='x', rotation=45)

# Chunk size distribution
df['chunk_length'].hist(bins=50, ax=axes[1, 0], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Chunk Size Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Chunk Length (characters)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df['chunk_length'].mean(), color='red', linestyle='--', label='Mean')
axes[1, 0].legend()

# Webpage vs regular docs
df['is_webpage'].value_counts().plot(kind='pie', ax=axes[1, 1], autopct='%1.1f%%', colors=['lightblue', 'orange'])
axes[1, 1].set_title('Webpage vs Regular Documents', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\n📊 Total Chunks: {len(df):,}")
print(f"📊 Average Chunk Size: {df['chunk_length'].mean():.0f} characters")
print(f"📊 Number of Insurance Providers: {df['insurance'].nunique()}")

## 8. Investigate Specific Chunks

In [ ]:
# Find chunks from a specific document
insurance = "goudse_expat_pakket"
doc_type = "conditions"

chunks = client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[
            FieldCondition(key="metadata.insurance_provider", match=MatchValue(value=insurance)),
            FieldCondition(key="metadata.document_type", match=MatchValue(value=doc_type)),
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)

print(f"📄 Found {len(chunks[0])} chunks for {insurance} - {doc_type}\n")
print("=" * 70)

for i, chunk in enumerate(chunks[0][:5], 1):
    meta = chunk.payload.get('metadata', {})
    content = chunk.payload.get('page_content', '')
    
    print(f"\nChunk {i}:")
    print(f"  Document: {meta.get('document_name', 'N/A')}")
    print(f"  Headers: {meta.get('header_1', '')} > {meta.get('header_2', '')} > {meta.get('header_3', '')}")
    print(f"  Chunk Index: {meta.get('chunk_index', 'N/A')}")
    print(f"  Length: {len(content)} chars")
    print(f"  Content Preview:\n    {content[:300]}...")
    print("-" * 70)

## 9. Debug: Raw Qdrant Search

In [ ]:
# Low-level Qdrant search (useful for debugging)
query = "pregnancy coverage"

# Get query embedding
query_vector = embeddings.embed_query(query)

print(f"Query: {query}")
print(f"Embedding dimension: {len(query_vector)}")
print(f"Embedding preview: {query_vector[:5]}...\n")

# Search directly with Qdrant client
results = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=5,
    with_payload=True
)

print("Raw Qdrant Results:")
print("=" * 70)
for i, hit in enumerate(results, 1):
    print(f"\n{i}. Score: {hit.score:.4f}")
    print(f"   ID: {hit.id}")
    print(f"   Metadata: {json.dumps(hit.payload.get('metadata', {}), indent=2)}")
    print(f"   Content: {hit.payload.get('page_content', '')[:150]}...")

## 10. Compare Models (if you indexed with multiple models)

In [ ]:
# If you have multiple collections from different models, compare them
query = "What is the coverage for dental care?"

# List collections that might be different models
model_collections = [
    "insurance_docs_text-embedding-3-large",
    "insurance_docs_text-embedding-3-small",
    "insurance_docs_google_text-embedding-004",
]

print(f"🔍 Comparing query across models: '{query}'\n")
print("=" * 70)

for col_name in model_collections:
    try:
        # Check if collection exists
        client.get_collection(col_name)
        
        print(f"\n📊 Model: {col_name.replace('insurance_docs_', '')}")
        print("-" * 70)
        
        # Create vector store for this collection
        vs = QdrantVectorStore(
            client=client,
            collection_name=col_name,
            embedding=embeddings,
        )
        
        results = vs.similarity_search_with_score(query, k=3)
        
        for i, (doc, score) in enumerate(results, 1):
            print(f"  {i}. Score: {score:.4f}")
            print(f"     Insurance: {doc.metadata.get('insurance_provider', 'N/A')}")
            print(f"     Preview: {doc.page_content[:100]}...\n")
            
    except Exception as e:
        print(f"  ⚠️  Collection not found: {col_name}")

## Tips & Tricks

### Investigation Tips:

1. **Check chunk quality:**
   - Look at `chunk_length` distribution - too small/large?
   - Verify headers are preserved in metadata
   - Check if sub-chunks have proper context

2. **Test different queries:**
   - Specific: "pregnancy complications coverage"
   - General: "what is covered"
   - Edge cases: "repatriation", "SOS"

3. **Use filters to debug:**
   - Filter by insurance to compare coverage
   - Filter by doc_type to check structure
   - Filter by is_webpage for latest info

4. **Compare embedding models:**
   - Index with multiple models (different collections)
   - Run same query on each
   - Compare average scores and relevance

### Visualization Ideas:

1. **Heatmap:** Query performance per insurance
2. **Word cloud:** Most common terms in chunks
3. **Scatter:** Chunk size vs score for a query
4. **Network:** Document relationships based on similarity

### Troubleshooting:

- **Low scores (< 0.5):** Query might be too vague or not in corpus
- **Wrong results:** Check if filters are correct
- **Missing data:** Verify insurance name in filters matches exactly
- **Slow queries:** Consider reducing `k` or using filters

### Next Steps:

1. Save good test queries for evaluation
2. Create a test set of queries with expected results
3. Benchmark different chunking strategies
4. Test with real user questions